# Analysing oscillator data with Polars

This notebook discovers the generated oscillator dataset, defines lazy Polars scans over its Parquet files, checks that the resulting tables have the expected grain, and provides model solutions for all five research questions in [`SCENARIO.md`](SCENARIO.md). The repository copy is intentionally output-free; execute it from top to bottom to produce local results.

Run this notebook with the repository's `oscillator` Pixi environment. If the ignored dataset is absent, generate it from the repository root with:

```bash
pixi run --environment oscillator generate-oscillator-data
```

## Data ingestion

### Setup

The path search supports kernels started from the repository root, `source-code`, or this notebook's directory. The manifest is the dataset contract: it records the version, layout, row counts, and generation parameters.

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl


def find_oscillator_directory() -> Path:
    candidates = (
        Path.cwd(),
        Path.cwd() / "oscillator",
        Path.cwd() / "source-code" / "oscillator",
    )
    for candidate in candidates:
        if (candidate / "generate_dataset.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate source-code/oscillator from the current directory."
    )


OSCILLATOR_DIR = find_oscillator_directory()
DATA_DIR = OSCILLATOR_DIR / "data"
MANIFEST_PATH = DATA_DIR / "manifest.json"

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Dataset not found. From the repository root, run: "
        "pixi run --environment oscillator generate-oscillator-data"
    )

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
{
    "dataset_id": manifest["dataset_id"],
    "generator_version": manifest["generator_version"],
    **manifest["summary"],
}

### Define lazy Parquet scans

`scan_parquet` records a query plan without loading the complete dataset. Hive partition discovery adds `damping_ratio` from directory names such as `damping_ratio=0.01`; in this environment Polars infers it as a floating-point value. The source path is retained so partition selection remains inspectable.

In [ ]:
runs = pl.scan_parquet((DATA_DIR / "runs.parquet").as_posix())
solver_batches = pl.scan_parquet(
    (DATA_DIR / "solver_batches.parquet").as_posix()
)
observations_glob = (
    DATA_DIR / "observations" / "*" / "*.parquet"
).as_posix()
observations = pl.scan_parquet(
    observations_glob,
    hive_partitioning=True,
    include_file_paths="source_file",
)

In [ ]:
observations.collect_schema()

### Check the ingestion contract

Check row counts before interpreting the data. Each observation must join to exactly one run, every run must contain its declared number of samples, and the path-derived damping ratio must agree with the authoritative value in `runs.parquet`. `collect` executes only the compact checks requested from each lazy plan.

In [ ]:
pl.DataFrame(
    {
        "relation": ["runs", "solver_batches", "observations"],
        "rows": [
            runs.select(pl.len()).collect().item(),
            solver_batches.select(pl.len()).collect().item(),
            observations.select(pl.len()).collect().item(),
        ],
    }
)

In [ ]:
(
    observations.group_by("damping_ratio")
    .agg(
        pl.len().alias("observations"),
        pl.col("source_file").n_unique().alias("files"),
    )
    .sort("damping_ratio")
    .collect()
)

In [ ]:
per_run_quality = (
    observations.group_by("run_id")
    .agg(pl.len().alias("observed_samples"))
    .join(
        runs.select("run_id", "sample_count", "completed"),
        on="run_id",
        how="inner",
    )
    .select(
        pl.len().alias("runs_checked"),
        (pl.col("observed_samples") != pl.col("sample_count"))
        .sum()
        .alias("sample_count_mismatches"),
        (~pl.col("completed")).sum().alias("incomplete_runs"),
    )
)

partition_mismatches = (
    observations.join(
        runs.select("run_id", "damping_ratio"),
        on="run_id",
        how="inner",
        suffix="_run",
    )
    .filter(
        (pl.col("damping_ratio") - pl.col("damping_ratio_run")).abs()
        > 1.0e-12
    )
    .select(pl.len().alias("partition_mismatches"))
)

solver_quality = solver_batches.select(
    (~pl.col("success")).sum().alias("failed_batches")
)

pl.concat(
    [
        per_run_quality.collect(),
        partition_mismatches.collect(),
        solver_quality.collect(),
    ],
    how="horizontal",
)

### Build an analysis-ready lazy query

Join observations to run metadata using `run_id`. Keep the path-derived partition value visible as `partition_damping_ratio`, but use `run_damping_ratio` for numerical analysis. The result remains lazy until `collect` requests a concrete result.

In [ ]:
analysis_ready = (
    observations.rename({"damping_ratio": "partition_damping_ratio"})
    .join(
        runs.select(
            "run_id",
            pl.col("damping_ratio").alias("run_damping_ratio"),
            "frequency_ratio",
            "natural_angular_frequency_rad_s",
            "drive_angular_frequency_rad_s",
            "drive_acceleration_m_s2",
            "samples_per_period",
            "cycles",
        ),
        on="run_id",
        how="inner",
    )
)

(
    analysis_ready.filter(pl.col("partition_damping_ratio") == 0.01)
    .sort("run_id", "sample_index")
    .head(10)
    .collect()
)

In [ ]:
print(
    observations.filter(pl.col("damping_ratio") == 0.01)
    .select("run_id", "sample_index", "displacement_m")
    .explain()
)

## Reference solutions: estimate response amplitude

The first research question asks what should count as response amplitude. The three subsections below provide complete reference implementations of the alternatives in `SCENARIO.md`. In practice, select the estimator that matches the scientific claim; use two or more only when their differences are themselves informative.

For a common comparison, all three solutions use the final five driving periods: periods 36 through 40, represented internally by zero-based period indices 35 through 39. The window is half-open, so the endpoint after period 40 is excluded. A late window reduces the early transient's influence, but it does **not** establish steady state; the second research question must compare successive windows.

Each subsection is intentionally self-contained, so executing every preview repeats some scans. For a much larger dataset, calculate the shared grouped statistics in one pass or materialize a compact window summary when comparing estimators.

In [ ]:
window_start_period = 35
window_periods = 5
window_stop_period = window_start_period + window_periods

measurement_window = analysis_ready.filter(
    (
        pl.col("sample_index")
        >= window_start_period * pl.col("samples_per_period")
    )
    & (
        pl.col("sample_index")
        < window_stop_period * pl.col("samples_per_period")
    )
)

per_run_window = measurement_window.group_by("run_id").agg(
    pl.len().alias("samples"),
    pl.col("samples_per_period").first(),
)
window_quality = per_run_window.select(
    pl.col("samples").min().alias("minimum_samples"),
    pl.col("samples").max().alias("maximum_samples"),
    (
        pl.col("samples")
        != window_periods * pl.col("samples_per_period")
    )
    .sum()
    .alias("incomplete_windows"),
)
phase_quality = (
    measurement_window.sort("run_id", "sample_index")
    .with_columns(
        pl.col("drive_phase_rad")
        .diff()
        .over("run_id")
        .alias("phase_step")
    )
    .filter(pl.col("phase_step").is_not_null())
    .select(
        (
            pl.col("phase_step")
            - 2.0 * math.pi / pl.col("samples_per_period")
        )
        .abs()
        .max()
        .alias("maximum_phase_step_error")
    )
)

pl.concat(
    [window_quality.collect(), phase_quality.collect()],
    how="horizontal",
)

### Half peak-to-peak amplitude

For displacement samples $x_i$, estimate amplitude as $\hat{A}_{\mathrm{pp}} = (\max_i x_i - \min_i x_i)/2$. This measures total excursion, including any remaining transient, and can underestimate extrema that fall between stored samples.

In [ ]:
group_keys = ["run_id", "run_damping_ratio", "frequency_ratio"]
result_columns = [
    *group_keys,
    "method",
    "amplitude_m",
    "residual_rms_m",
]

amplitude_peak_to_peak = (
    measurement_window.group_by(group_keys)
    .agg(
        (
            (pl.col("displacement_m").max()
             - pl.col("displacement_m").min())
            / 2.0
        ).alias("amplitude_m")
    )
    .with_columns(
        pl.lit("peak_to_peak").alias("method"),
        pl.lit(None, dtype=pl.Float64).alias("residual_rms_m"),
    )
    .select(result_columns)
)

amplitude_peak_to_peak.sort(group_keys).head(10).collect()

### Demeaned root-mean-square amplitude

For $n$ displacement samples with mean $\bar{x}$, estimate amplitude as $\hat{A}_{\mathrm{RMS}} = \sqrt{(2/n)\sum_i(x_i-\bar{x})^2}$. The aggregate form below is equivalent. It is exact for a single sinusoid sampled over complete periods, but treats all oscillatory components as response.

In [ ]:
amplitude_rms = (
    measurement_window.group_by(group_keys)
    .agg(
        (
            2.0
            * (
                pl.col("displacement_m").pow(2).mean()
                - pl.col("displacement_m").mean().pow(2)
            )
        )
        .clip(lower_bound=0.0)
        .sqrt()
        .alias("amplitude_m")
    )
    .with_columns(
        pl.lit("demeaned_rms").alias("method"),
        pl.lit(None, dtype=pl.Float64).alias("residual_rms_m"),
    )
    .select(result_columns)
)

amplitude_rms.sort(group_keys).head(10).collect()

### Harmonic least-squares amplitude

Project displacement onto the known driving phase $\phi_i$ using $x_i = b_0 + b_c\cos(\phi_i) + b_s\sin(\phi_i) + \epsilon_i$. Under the checked assumptions—uniform phase spacing, complete periods, a half-open window, complete samples, and equal weights—the least-squares coefficients reduce to grouped sums. The drive-frequency amplitude is $A = \sqrt{b_c^2+b_s^2}$; the residual RMS measures motion not explained by that harmonic.

In [ ]:
sufficient_statistics = measurement_window.group_by(group_keys).agg(
    pl.col("displacement_m").mean().alias("b_0"),
    (
        2.0
        * (
            pl.col("displacement_m")
            * pl.col("drive_phase_rad").cos()
        ).mean()
    ).alias("b_c"),
    (
        2.0
        * (
            pl.col("displacement_m")
            * pl.col("drive_phase_rad").sin()
        ).mean()
    ).alias("b_s"),
    pl.col("displacement_m").pow(2).mean().alias("mean_x_squared"),
)

amplitude_harmonic = (
    sufficient_statistics.with_columns(
        (pl.col("b_c").pow(2) + pl.col("b_s").pow(2))
        .sqrt()
        .alias("amplitude_m"),
        (
            pl.col("mean_x_squared")
            - pl.col("b_0").pow(2)
            - (pl.col("b_c").pow(2) + pl.col("b_s").pow(2)) / 2.0
        )
        .clip(lower_bound=0.0)
        .sqrt()
        .alias("residual_rms_m"),
    )
    .with_columns(pl.lit("harmonic_least_squares").alias("method"))
    .select(result_columns)
)

amplitude_harmonic.sort(group_keys).head(10).collect()

### Compare the reference estimates

Combining the three compatible run-level relations makes differences visible without implying that every analysis should retain all three. The preview uses runs at the nominal resonance frequency ratio $r=1$.

In [ ]:
amplitude_estimates = pl.concat(
    [amplitude_peak_to_peak, amplitude_rms, amplitude_harmonic],
    how="vertical",
)

(
    amplitude_estimates.filter(
        (pl.col("frequency_ratio") - 1.0).abs() < 1.0e-12
    )
    .sort("run_damping_ratio", "method")
    .collect()
)

## Model solution: assess whether runs have settled

The second research question requires evidence from successive windows, not merely a late amplitude estimate. This model solution uses the harmonic least-squares estimator because it provides both the drive-frequency amplitude and a residual measure. It divides every run into eight half-open windows of five driving periods, compares the final two, and retains every run in an explicit settling class.

### Estimate amplitude in successive windows

The terminal sample is excluded before assigning zero-based window identifiers. Each window therefore contains exactly 250 observations: five periods with 50 samples per period. The same grouped harmonic projection used above is then applied independently to every run and window. The compact window-level result is collected once and converted back to a lazy frame because several later diagnostics reuse it; this avoids rescanning the observation files for every display.

In [ ]:
settling_window_periods = 5
window_group_keys = [
    "run_id",
    "run_damping_ratio",
    "frequency_ratio",
    "natural_angular_frequency_rad_s",
    "drive_acceleration_m_s2",
    "window_id",
]

windowed = (
    analysis_ready.filter(
        pl.col("sample_index")
        < pl.col("cycles") * pl.col("samples_per_period")
    ).with_columns(
        (
            pl.col("sample_index")
            / (
                pl.col("samples_per_period")
                * settling_window_periods
            )
        )
        .floor()
        .cast(pl.Int32)
        .alias("window_id")
    )
)

window_statistics = windowed.group_by(window_group_keys).agg(
    pl.len().alias("samples"),
    pl.col("displacement_m").mean().alias("b_0"),
    (
        2.0
        * (
            pl.col("displacement_m")
            * pl.col("drive_phase_rad").cos()
        ).mean()
    ).alias("b_c"),
    (
        2.0
        * (
            pl.col("displacement_m")
            * pl.col("drive_phase_rad").sin()
        ).mean()
    ).alias("b_s"),
    pl.col("displacement_m").pow(2).mean().alias("mean_x_squared"),
)

harmonic_by_window = (
    window_statistics.with_columns(
        (pl.col("b_c").pow(2) + pl.col("b_s").pow(2))
        .sqrt()
        .alias("amplitude_m"),
        (
            pl.col("mean_x_squared")
            - pl.col("b_0").pow(2)
            - (pl.col("b_c").pow(2) + pl.col("b_s").pow(2)) / 2.0
        )
        .clip(lower_bound=0.0)
        .sqrt()
        .alias("residual_rms_m"),
    )
    .with_columns(
        (pl.col("window_id") * settling_window_periods + 1)
        .alias("first_period"),
        ((pl.col("window_id") + 1) * settling_window_periods)
        .alias("last_period"),
        (pl.col("residual_rms_m") / pl.col("amplitude_m"))
        .alias("residual_fraction"),
    )
    .select(
        *window_group_keys,
        "first_period",
        "last_period",
        "samples",
        "amplitude_m",
        "residual_rms_m",
        "residual_fraction",
    )
    .collect()
    .lazy()
)

(
    harmonic_by_window.filter(
        (pl.col("run_damping_ratio") - 0.01).abs() < 1.0e-12,
        (pl.col("frequency_ratio") - 1.0).abs() < 1.0e-12,
    )
    .sort("window_id")
    .collect()
)

### Compare the final two windows

Order windows within each run, retain the preceding amplitude with `shift`, and select the final window. For amplitudes $A_1$ and $A_2$, the symmetric relative change is $\Delta_A = 2|A_2-A_1|/(A_1+A_2)$. Unlike division by only one amplitude, this definition treats the two windows symmetrically.

In [ ]:
ordered_windows = harmonic_by_window.sort("run_id", "window_id")
amplitude_sum = (
    pl.col("amplitude_m") + pl.col("previous_amplitude_m")
)

late_window_comparison = (
    ordered_windows.with_columns(
        pl.col("window_id")
        .shift(1)
        .over("run_id")
        .alias("previous_window_id"),
        pl.col("amplitude_m")
        .shift(1)
        .over("run_id")
        .alias("previous_amplitude_m"),
        pl.col("window_id")
        .max()
        .over("run_id")
        .alias("final_window_id"),
    )
    .filter(pl.col("window_id") == pl.col("final_window_id"))
    .with_columns(
        pl.when(amplitude_sum > 0.0)
        .then(
            2.0
            * (
                pl.col("amplitude_m")
                - pl.col("previous_amplitude_m")
            ).abs()
            / amplitude_sum
        )
        .otherwise(None)
        .alias("relative_amplitude_change")
    )
    .select(
        "run_id",
        "run_damping_ratio",
        "frequency_ratio",
        "natural_angular_frequency_rad_s",
        "drive_acceleration_m_s2",
        "previous_window_id",
        "final_window_id",
        "previous_amplitude_m",
        pl.col("amplitude_m").alias("final_amplitude_m"),
        "relative_amplitude_change",
        pl.col("residual_rms_m").alias("final_residual_rms_m"),
        pl.col("residual_fraction").alias("final_residual_fraction"),
    )
)

(
    late_window_comparison.sort(
        "relative_amplitude_change", descending=True
    )
    .head(10)
    .collect()
)

### Classify without discarding runs

This model solution uses 1% as both the maximum acceptable amplitude change and the maximum residual-to-amplitude ratio. These are transparent demonstration criteria, not universal physical constants. A run is `still_changing` when its late amplitudes differ by more than 1%; `stable_amplitude_with_residual` when its amplitude is stable but more than 1% of it remains as residual RMS; and `stable_response` only when both criteria pass. The later sensitivity investigation should vary both thresholds.

In [ ]:
amplitude_change_tolerance = 0.01
residual_fraction_tolerance = 0.01
total_cycles = manifest["parameters"]["cycles"]
transient_envelope_target = 0.01

settling_assessment = late_window_comparison.with_columns(
    pl.when(
        pl.col("relative_amplitude_change").is_null()
        | pl.col("final_residual_fraction").is_null()
    )
    .then(pl.lit("insufficient_data"))
    .when(
        pl.col("relative_amplitude_change")
        > amplitude_change_tolerance
    )
    .then(pl.lit("still_changing"))
    .when(
        pl.col("final_residual_fraction")
        > residual_fraction_tolerance
    )
    .then(pl.lit("stable_amplitude_with_residual"))
    .otherwise(pl.lit("stable_response"))
    .alias("settling_class"),
    (
        -2.0
        * math.pi
        * pl.col("run_damping_ratio")
        * total_cycles
        / pl.col("frequency_ratio")
    )
    .exp()
    .alias("expected_transient_factor"),
    (
        (
            -pl.col("frequency_ratio")
            * math.log(transient_envelope_target)
            / (
                2.0
                * math.pi
                * pl.col("run_damping_ratio")
            )
        )
        .ceil()
        .cast(pl.Int32)
        - total_cycles
    )
    .clip(lower_bound=0)
    .alias("estimated_additional_periods"),
)

(
    settling_assessment.group_by(
        "run_damping_ratio", "settling_class"
    )
    .agg(pl.len().alias("runs"))
    .sort("run_damping_ratio", "settling_class")
    .collect()
)

### Compare with the transient-decay expectation

The homogeneous transient envelope is expected to decay by a factor $E_N = \exp(-2\pi\zeta N/r)$ after $N$ driving periods, where $\zeta$ is the damping ratio and $r$ is the driving-to-natural frequency ratio. Solving $E_N \leq \epsilon$ gives the planning estimate $N_\epsilon = \lceil-r\ln(\epsilon)/(2\pi\zeta)\rceil$. Here $\epsilon=0.01$. Neither $E_N$ nor $N_\epsilon$ is a measured residual or proof of convergence: the unknown initial transient magnitude and phase still matter. The summary checks whether the observed classifications follow the expected damping pattern; the row-level preview then identifies unresolved runs and gives a rough estimate of how many additional periods may be needed.

In [ ]:
(
    settling_assessment.group_by("run_damping_ratio").agg(
        pl.col("expected_transient_factor")
        .min()
        .alias("minimum_expected_transient_factor"),
        pl.col("expected_transient_factor")
        .max()
        .alias("maximum_expected_transient_factor"),
        pl.col("relative_amplitude_change")
        .max()
        .alias("maximum_observed_amplitude_change"),
        (pl.col("settling_class") != "stable_response")
        .sum()
        .alias("unresolved_runs"),
    )
    .sort("run_damping_ratio")
    .collect()
)

Keep the unresolved runs in the result. Extending each simulation by at least the estimated number of periods, then repeating the adjacent-window test, is a defensible next step; it is not valid to reclassify a run from the estimate alone.

In [ ]:
(
    settling_assessment.filter(
        pl.col("settling_class") != "stable_response"
    )
    .sort("relative_amplitude_change", descending=True)
    .head(10)
    .collect()
)

## Model solution: quantify how damping changes resonance

This reference solution treats only `stable_response` runs as defensible for estimating a resonance peak or its width. That conservative choice avoids presenting a transient-contaminated peak as a steady-state result, but it can leave a damping ratio without a defensible peak. All runs remain in the analysis and visualization so that this loss of coverage is visible rather than silently filtered away.

### Calculate the dimensionless gain

For final-window harmonic amplitude $A$, natural angular frequency $\omega_0$, and driving-acceleration amplitude $a$, the dimensionless gain is $G=A\omega_0^2/a$. The compact run-level table retains the settling evidence beside $G$ so every subsequent claim can be traced back to its convergence assessment.

In [ ]:
resonance_response = (
    settling_assessment.with_columns(
        pl.when(pl.col("drive_acceleration_m_s2") != 0.0)
        .then(
            pl.col("final_amplitude_m")
            * pl.col("natural_angular_frequency_rad_s").pow(2)
            / pl.col("drive_acceleration_m_s2")
        )
        .otherwise(None)
        .alias("gain")
    )
    .collect()
    .lazy()
)

(
    resonance_response.group_by(
        "run_damping_ratio", "settling_class"
    )
    .agg(pl.len().alias("runs"))
    .sort("run_damping_ratio", "settling_class")
    .collect()
)

### Compare the resonance curves

Each panel uses the same frequency-ratio range but an independent gain scale, because a shared vertical scale would flatten the more strongly damped responses. The pale line connects sampled frequencies only; no smoothing or interpolation is used. Marker colour identifies whether each point supports a steady-state claim.

In [ ]:
response_for_plot = (
    resonance_response.select(
        "run_damping_ratio",
        "frequency_ratio",
        "gain",
        "settling_class",
    )
    .sort("run_damping_ratio", "frequency_ratio")
    .collect()
    .to_pandas()
)

status_styles = [
    ("stable_response", "tab:blue", "stable response"),
    (
        "stable_amplitude_with_residual",
        "tab:orange",
        "stable amplitude, residual remains",
    ),
    ("still_changing", "tab:red", "still changing"),
]
damping_ratios = sorted(response_for_plot["run_damping_ratio"].unique())
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)

for ax, damping_ratio in zip(axes.flat, damping_ratios, strict=True):
    response = response_for_plot[
        response_for_plot["run_damping_ratio"] == damping_ratio
    ]
    ax.plot(
        response["frequency_ratio"],
        response["gain"],
        color="0.75",
        linewidth=1.0,
        zorder=1,
    )
    for settling_class, colour, label in status_styles:
        points = response[response["settling_class"] == settling_class]
        ax.scatter(
            points["frequency_ratio"],
            points["gain"],
            s=14,
            color=colour,
            label=label if ax is axes.flat[0] else None,
            zorder=2,
        )
    ax.set_title(f"damping ratio = {damping_ratio:g}")
    ax.grid(alpha=0.2)

fig.supxlabel("frequency ratio r", y=0.08)
fig.supylabel("dimensionless gain G")
fig.legend(
    *axes.flat[0].get_legend_handles_labels(),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=3,
)
fig.tight_layout(rect=(0.0, 0.13, 1.0, 1.0))

### Locate each observed peak

The observed peak is the largest sampled gain for a damping ratio; ties are resolved toward the smaller frequency ratio. Its settling class, late-window amplitude change, and residual fraction are reported with the peak. `peak_defensible` is therefore an evidence flag, not another numerical threshold introduced after seeing the curve. The peak frequency is reported on the sampled grid and is not refined by interpolation.

In [ ]:
coverage = resonance_response.group_by("run_damping_ratio").agg(
    pl.len().alias("total_runs"),
    (pl.col("settling_class") == "stable_response")
    .sum()
    .alias("stable_runs"),
)

peak_rows = (
    resonance_response.sort(
        ["run_damping_ratio", "gain", "frequency_ratio"],
        descending=[False, True, False],
    )
    .group_by("run_damping_ratio", maintain_order=True)
    .first()
)

resonance_peaks = (
    peak_rows.join(coverage, on="run_damping_ratio")
    .select(
        "run_damping_ratio",
        "total_runs",
        "stable_runs",
        pl.col("frequency_ratio")
        .alias("observed_peak_frequency_ratio"),
        pl.col("gain").alias("observed_peak_gain"),
        pl.col("settling_class").alias("peak_settling_class"),
        pl.col("relative_amplitude_change")
        .alias("peak_relative_amplitude_change"),
        pl.col("final_residual_fraction")
        .alias("peak_residual_fraction"),
        (pl.col("settling_class") == "stable_response")
        .alias("peak_defensible"),
        (pl.col("gain") / math.sqrt(2.0))
        .alias("half_power_gain"),
    )
    .sort("run_damping_ratio")
    .collect()
    .lazy()
)

resonance_peaks.select(pl.exclude("half_power_gain")).collect()

### Estimate width and sharpness at grid resolution

For a defensible peak $G_\mathrm{max}$, the half-power threshold is $G_\mathrm{max}/\sqrt{2}$. Let $r_\mathrm{L,in}$ and $r_\mathrm{R,in}$ be the smallest and largest sampled frequency ratios at or above that threshold. If both crossings are bracketed by the grid, the half-power bandwidth $\Delta r$ lies between $r_\mathrm{R,in}-r_\mathrm{L,in}$ and that span plus two grid steps. The corresponding sharpness is reported as a quality-factor interval $Q=r_\mathrm{peak}/\Delta r$, where $r_\mathrm{peak}$ is the sampled peak frequency ratio. This interval assumes a single contiguous peak with monotonic threshold crossings; no sub-grid interpolation is claimed. A missing interval is a scientific result: either the peak is unsettled or the sampled frequency range does not bracket both crossings.

In [ ]:
frequency_grid_spacing = (
    manifest["parameters"]["frequency_max"]
    - manifest["parameters"]["frequency_min"]
) / (manifest["parameters"]["frequency_points"] - 1)

half_power_points = (
    resonance_response.join(
        resonance_peaks.select(
            "run_damping_ratio",
            "peak_defensible",
            "half_power_gain",
        ),
        on="run_damping_ratio",
    )
    .filter(
        pl.col("peak_defensible")
        & (pl.col("settling_class") == "stable_response")
        & (pl.col("gain") >= pl.col("half_power_gain"))
    )
    .group_by("run_damping_ratio")
    .agg(
        pl.col("frequency_ratio").min().alias("lower_inside_ratio"),
        pl.col("frequency_ratio").max().alias("upper_inside_ratio"),
        pl.len().alias("half_power_samples"),
    )
)

grid_bounds = resonance_response.group_by("run_damping_ratio").agg(
    pl.col("frequency_ratio").min().alias("grid_minimum_ratio"),
    pl.col("frequency_ratio").max().alias("grid_maximum_ratio"),
)

resonance_bandwidth = (
    resonance_peaks.select(
        "run_damping_ratio",
        "observed_peak_frequency_ratio",
        "peak_defensible",
    )
    .join(grid_bounds, on="run_damping_ratio")
    .join(half_power_points, on="run_damping_ratio", how="left")
    .with_columns(
        (pl.col("upper_inside_ratio") - pl.col("lower_inside_ratio"))
        .alias("sampled_half_power_span"),
        pl.when(~pl.col("peak_defensible"))
        .then(pl.lit("peak_not_settled"))
        .when(pl.col("lower_inside_ratio").is_null())
        .then(pl.lit("no_stable_half_power_points"))
        .when(
            (
                (
                    pl.col("lower_inside_ratio")
                    - pl.col("grid_minimum_ratio")
                ).abs()
                < 1.0e-12
            )
            | (
                (
                    pl.col("upper_inside_ratio")
                    - pl.col("grid_maximum_ratio")
                ).abs()
                < 1.0e-12
            )
        )
        .then(pl.lit("not_bracketed_by_frequency_grid"))
        .otherwise(pl.lit("bracketed_to_grid_resolution"))
        .alias("bandwidth_status"),
    )
    .with_columns(
        pl.when(
            pl.col("bandwidth_status")
            == "bracketed_to_grid_resolution"
        )
        .then(pl.col("sampled_half_power_span"))
        .otherwise(None)
        .alias("half_power_width_lower"),
        pl.when(
            pl.col("bandwidth_status")
            == "bracketed_to_grid_resolution"
        )
        .then(
            pl.col("sampled_half_power_span")
            + 2.0 * frequency_grid_spacing
        )
        .otherwise(None)
        .alias("half_power_width_upper"),
    )
    .with_columns(
        (
            pl.col("observed_peak_frequency_ratio")
            / pl.col("half_power_width_upper")
        ).alias("quality_factor_lower"),
        (
            pl.col("observed_peak_frequency_ratio")
            / pl.col("half_power_width_lower")
        ).alias("quality_factor_upper"),
    )
    .select(
        "run_damping_ratio",
        "bandwidth_status",
        "lower_inside_ratio",
        "upper_inside_ratio",
        "half_power_samples",
        "half_power_width_lower",
        "half_power_width_upper",
        "quality_factor_lower",
        "quality_factor_upper",
    )
    .sort("run_damping_ratio")
    .collect()
    .lazy()
)

resonance_bandwidth.collect()

The model solution supports the qualitative conclusion that increasing damping lowers and broadens the resonance response. It does **not** support a defensible peak or width for the least-damped ensemble because its observed peak has not settled. At the strongest damping, the peak is defensible but the lower half-power crossing lies outside the sampled frequency range, so the full bandwidth and $Q$ remain unresolved. These limitations should be carried into later comparisons rather than replaced by interpolated values.

## Model solution: compare with steady-state theory

The simulation and analytical expression describe the same idealized linear oscillator. Agreement is therefore a consistency check on the ODE integration, data reduction, grouping, and normalization—not independent experimental validation. Disagreement must first be examined for transient contamination and window dependence before it is attributed to the model or solver.

### Calculate the theoretical gain

For frequency ratio $r=\omega_\mathrm{d}/\omega_0$ and damping ratio $\zeta$, the steady-state dimensionless gain is $G_\mathrm{theory}=[(1-r^2)^2+(2\zeta r)^2]^{-1/2}$, where $\omega_\mathrm{d}$ is the driving angular frequency and $\omega_0$ is the natural angular frequency. Define the signed difference as $d=G_\mathrm{estimated}-G_\mathrm{theory}$ and the relative difference as $\delta=d/G_\mathrm{theory}$. The sampled parameter grid has $\zeta>0$, so $G_\mathrm{theory}$ is finite and positive; the diagnostic below verifies that its denominator is safely away from zero before relative differences are interpreted.

In [ ]:
theoretical_gain = (
    1.0
    / (
        (1.0 - pl.col("frequency_ratio").pow(2)).pow(2)
        + (
            2.0
            * pl.col("run_damping_ratio")
            * pl.col("frequency_ratio")
        ).pow(2)
    ).sqrt()
).alias("theoretical_gain")

theory_comparison = (
    resonance_response.with_columns(theoretical_gain)
    .with_columns(
        (pl.col("gain") - pl.col("theoretical_gain"))
        .alias("signed_gain_difference")
    )
    .with_columns(
        (
            pl.col("signed_gain_difference")
            / pl.col("theoretical_gain")
        ).alias("relative_gain_difference")
    )
    .collect()
    .lazy()
)

theory_comparison.select(
    pl.col("theoretical_gain").min()
    .alias("minimum_theoretical_gain"),
    pl.col("theoretical_gain").max()
    .alias("maximum_theoretical_gain"),
    pl.col("relative_gain_difference").null_count()
    .alias("undefined_relative_differences"),
).collect()

### Compare the final-window estimates

Summarize discrepancies separately by damping ratio and settling class. The median absolute relative difference describes a typical discrepancy, while the maximum exposes localized failures that a median could hide. The second table deliberately focuses on the resonance neighbourhood $|r-1|\leq0.2$, where both the gain and sensitivity to incomplete settling are largest.

In [ ]:
display(
    theory_comparison.group_by(
        "run_damping_ratio", "settling_class"
    )
    .agg(
        pl.len().alias("runs"),
        pl.col("signed_gain_difference").mean()
        .alias("mean_signed_gain_difference"),
        pl.col("relative_gain_difference").abs().median()
        .alias("median_absolute_relative_difference"),
        pl.col("relative_gain_difference").abs().max()
        .alias("maximum_absolute_relative_difference"),
    )
    .sort("run_damping_ratio", "settling_class")
    .collect()
)

(
    theory_comparison.filter(
        (pl.col("frequency_ratio") - 1.0).abs() <= 0.2
    )
    .with_columns(
        pl.col("relative_gain_difference")
        .abs()
        .alias("_absolute_relative_difference")
    )
    .sort("_absolute_relative_difference", descending=True)
    .head(12)
    .select(
        "run_damping_ratio",
        "frequency_ratio",
        pl.col("gain").alias("estimated_gain"),
        "theoretical_gain",
        "signed_gain_difference",
        "relative_gain_difference",
        "settling_class",
    )
    .collect()
)

### Locate discrepancies across frequency

The panels show $100\delta$, the signed relative difference in percent, with zero denoting agreement. Independent vertical scales keep small discrepancies visible for the well-settled ensembles; they therefore must not be used to compare apparent vertical magnitudes between panels. Points remain classified by settling evidence, and no smoothing is applied.

In [ ]:
error_for_plot = (
    theory_comparison.select(
        "run_damping_ratio",
        "frequency_ratio",
        (100.0 * pl.col("relative_gain_difference"))
        .alias("relative_gain_difference_percent"),
        "settling_class",
    )
    .sort("run_damping_ratio", "frequency_ratio")
    .collect()
    .to_pandas()
)

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
for ax, damping_ratio in zip(axes.flat, damping_ratios, strict=True):
    errors = error_for_plot[
        error_for_plot["run_damping_ratio"] == damping_ratio
    ]
    ax.axhline(0.0, color="0.5", linewidth=0.8)
    ax.plot(
        errors["frequency_ratio"],
        errors["relative_gain_difference_percent"],
        color="0.75",
        linewidth=1.0,
        zorder=1,
    )
    for settling_class, colour, label in status_styles:
        points = errors[errors["settling_class"] == settling_class]
        ax.scatter(
            points["frequency_ratio"],
            points["relative_gain_difference_percent"],
            s=14,
            color=colour,
            label=label if ax is axes.flat[0] else None,
            zorder=2,
        )
    ax.set_title(f"damping ratio = {damping_ratio:g}")
    ax.grid(alpha=0.2)

fig.supxlabel("frequency ratio r", y=0.08)
fig.supylabel("signed relative gain difference (%)")
fig.legend(
    *axes.flat[0].get_legend_handles_labels(),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=3,
)
fig.tight_layout(rect=(0.0, 0.13, 1.0, 1.0))

### Test the comparison across analysis windows

A final-window comparison alone cannot distinguish persistent model disagreement from a transient that is still decaying. Recalculate the gain difference for every five-period window, then summarize the median and worst absolute relative difference across the sampled frequency ratios. The plot uses the maximum deliberately: it asks whether any part of the resonance curve remains poorly represented, rather than allowing well-behaved off-resonance runs to dominate an average.

In [ ]:
window_theory_comparison = (
    harmonic_by_window.with_columns(
        (
            pl.col("amplitude_m")
            * pl.col("natural_angular_frequency_rad_s").pow(2)
            / pl.col("drive_acceleration_m_s2")
        ).alias("estimated_gain"),
        theoretical_gain,
    )
    .with_columns(
        (
            pl.col("estimated_gain")
            - pl.col("theoretical_gain")
        ).alias("signed_gain_difference")
    )
    .with_columns(
        (
            pl.col("signed_gain_difference")
            / pl.col("theoretical_gain")
        ).alias("relative_gain_difference")
    )
)

window_error_summary = (
    window_theory_comparison.group_by(
        "run_damping_ratio",
        "window_id",
        "first_period",
        "last_period",
    )
    .agg(
        pl.col("relative_gain_difference").abs().median()
        .alias("median_absolute_relative_difference"),
        pl.col("relative_gain_difference").abs().max()
        .alias("maximum_absolute_relative_difference"),
    )
    .sort("run_damping_ratio", "window_id")
    .collect()
    .lazy()
)

display(
    window_error_summary.filter(pl.col("window_id").is_in([0, 7]))
    .sort("run_damping_ratio", "window_id")
    .collect()
)

window_errors_for_plot = window_error_summary.collect().to_pandas()
fig, ax = plt.subplots(figsize=(8, 5))
for damping_ratio in damping_ratios:
    errors = window_errors_for_plot[
        window_errors_for_plot["run_damping_ratio"]
        == damping_ratio
    ]
    ax.plot(
        errors["last_period"],
        100.0 * errors["maximum_absolute_relative_difference"],
        marker="o",
        label=f"damping ratio = {damping_ratio:g}",
    )
ax.set_yscale("log")
ax.set_xlabel("last period in five-period window")
ax.set_ylabel("maximum absolute relative difference (%)")
ax.grid(alpha=0.2, which="both")
ax.legend(ncol=2)
fig.tight_layout()

For $\zeta\geq0.03$, every run is classified as stable and the largest final-window relative discrepancy falls from about 0.13% at $\zeta=0.03$ to the numerical-noise scale for the more strongly damped ensembles. The $\zeta=0.01$ result is more instructive: the worst discrepancy is about 9.5% near resonance, and even the provisional `stable_response` subgroup contains discrepancies of about 6.9%. Two adjacent windows can therefore appear stable while a slowly decaying oscillatory transient remains. The discrepancies shrink across successive windows, so the pattern supports transient contamination rather than a failure of the steady-state expression. Agreement still does not independently validate the physical model because both calculations encode the same linear oscillator assumptions; instead, this comparison exposes a limitation of the current settling rule that should be tested in the sensitivity analysis.

## Model solution: test sensitivity to analytical choices

A sensitivity analysis should target choices capable of changing the scientific conclusion. This solution first varies the settling criteria, which determine which runs support a steady-state claim, and then compares the three amplitude estimators, which summarize different aspects of the motion. The objective is not to select whichever result is most convenient, but to identify conclusions shared by credible alternatives and expose those that remain conditional.

### Vary the settling tolerances

Let $\tau_A$ be the maximum accepted relative amplitude change between the final two windows and $\tau_R$ the maximum accepted residual-to-amplitude ratio. Compare paired strict, baseline, and permissive criteria with $\tau_A=\tau_R$ equal to 0.5%, 1%, and 2%, respectively. Varying both together is deliberately compact, but its disadvantage is that a changed classification cannot be attributed to one threshold. The peak assessment asks whether the largest observed gain becomes defensible under any credible scenario.

In [ ]:
pl.Config.set_tbl_rows(30)

tolerance_scenarios = pl.DataFrame(
    {
        "scenario_order": [1, 2, 3],
        "scenario": ["strict", "baseline", "permissive"],
        "amplitude_change_tolerance": [0.005, 0.010, 0.020],
        "residual_fraction_tolerance": [0.005, 0.010, 0.020],
    }
).lazy()

tolerance_sensitivity = (
    late_window_comparison.join(tolerance_scenarios, how="cross")
    .with_columns(
        pl.when(
            pl.col("relative_amplitude_change").is_null()
            | pl.col("final_residual_fraction").is_null()
        )
        .then(pl.lit("insufficient_data"))
        .when(
            pl.col("relative_amplitude_change")
            > pl.col("amplitude_change_tolerance")
        )
        .then(pl.lit("still_changing"))
        .when(
            pl.col("final_residual_fraction")
            > pl.col("residual_fraction_tolerance")
        )
        .then(pl.lit("stable_amplitude_with_residual"))
        .otherwise(pl.lit("stable_response"))
        .alias("sensitivity_settling_class")
    )
    .collect()
    .lazy()
)

display(
    tolerance_sensitivity.group_by(
        "scenario_order",
        "scenario",
        "amplitude_change_tolerance",
        "residual_fraction_tolerance",
        "run_damping_ratio",
    )
    .agg(
        (
            pl.col("sensitivity_settling_class")
            == "stable_response"
        )
        .sum()
        .alias("stable_response_runs"),
        (
            pl.col("sensitivity_settling_class")
            == "stable_amplitude_with_residual"
        )
        .sum()
        .alias("residual_runs"),
        (
            pl.col("sensitivity_settling_class")
            == "still_changing"
        )
        .sum()
        .alias("changing_runs"),
    )
    .sort("scenario_order", "run_damping_ratio")
    .select(pl.exclude("scenario_order"))
    .collect()
)

peak_runs = (
    resonance_response.sort(
        ["run_damping_ratio", "gain", "frequency_ratio"],
        descending=[False, True, False],
    )
    .group_by("run_damping_ratio", maintain_order=True)
    .first()
    .select(
        "run_id",
        pl.col("frequency_ratio")
        .alias("observed_peak_frequency_ratio"),
        pl.col("gain").alias("observed_peak_gain"),
    )
)

(
    tolerance_sensitivity.join(peak_runs, on="run_id")
    .select(
        "scenario_order",
        "scenario",
        "run_damping_ratio",
        "observed_peak_frequency_ratio",
        "observed_peak_gain",
        pl.col("sensitivity_settling_class")
        .alias("peak_settling_class"),
        (
            pl.col("sensitivity_settling_class")
            == "stable_response"
        ).alias("peak_defensible"),
    )
    .sort("scenario_order", "run_damping_ratio")
    .select(pl.exclude("scenario_order"))
    .collect()
)

### Compare the amplitude estimators

Compare half peak-to-peak amplitude, demeaned RMS amplitude, and harmonic least-squares amplitude over the same final five-period window. For estimator $m$, define $\Delta_m=(A_m-A_\mathrm{H})/A_\mathrm{H}$, where $A_m$ is its amplitude and $A_\mathrm{H}$ is the harmonic estimate for the same run. The harmonic estimate is the reference because it isolates the drive-frequency component; the alternatives include transient or other residual motion and are therefore diagnostic rather than automatically interchangeable. Report both observed peaks and estimator differences, retaining the baseline settling class.

In [ ]:
estimator_sensitivity = (
    amplitude_estimates.join(
        settling_assessment.select(
            "run_id",
            "natural_angular_frequency_rad_s",
            "drive_acceleration_m_s2",
            "settling_class",
        ),
        on="run_id",
    )
    .with_columns(
        (
            pl.col("amplitude_m")
            * pl.col("natural_angular_frequency_rad_s").pow(2)
            / pl.col("drive_acceleration_m_s2")
        ).alias("gain")
    )
    .collect()
    .lazy()
)

estimator_peaks = (
    estimator_sensitivity.sort(
        [
            "method",
            "run_damping_ratio",
            "gain",
            "frequency_ratio",
        ],
        descending=[False, False, True, False],
    )
    .group_by("method", "run_damping_ratio", maintain_order=True)
    .first()
    .select(
        "method",
        "run_damping_ratio",
        pl.col("frequency_ratio")
        .alias("observed_peak_frequency_ratio"),
        pl.col("gain").alias("observed_peak_gain"),
        pl.col("settling_class").alias("peak_settling_class"),
        (pl.col("settling_class") == "stable_response")
        .alias("peak_defensible"),
    )
    .sort("run_damping_ratio", "method")
    .collect()
    .lazy()
)

display(estimator_peaks.collect())

harmonic_reference = estimator_sensitivity.filter(
    pl.col("method") == "harmonic_least_squares"
).select(
    "run_id",
    pl.col("amplitude_m").alias("harmonic_amplitude_m"),
)

estimator_agreement = (
    estimator_sensitivity.join(harmonic_reference, on="run_id")
    .filter(pl.col("settling_class") == "stable_response")
    .with_columns(
        (
            (pl.col("amplitude_m") - pl.col("harmonic_amplitude_m"))
            / pl.col("harmonic_amplitude_m")
        )
        .abs()
        .alias("absolute_relative_difference")
    )
    .group_by("run_damping_ratio", "method")
    .agg(
        pl.len().alias("baseline_stable_runs"),
        pl.col("absolute_relative_difference").median()
        .alias("median_absolute_relative_difference"),
        pl.col("absolute_relative_difference").max()
        .alias("maximum_absolute_relative_difference"),
    )
    .sort("run_damping_ratio", "method")
    .collect()
)

display(estimator_agreement)
pl.Config.restore_defaults()

Some conclusions are robust across these analytical choices. Increasing the damping ratio $\zeta$ lowers and broadens the resonance, and the observed peak for $\zeta = 0.01$ remains classified as `still_changing` under every tolerance scenario and with every amplitude estimator. There is therefore no defensible range of peak values to report for that ensemble: the result is unresolved. For $\zeta \geq 0.03$, the observed peak remains defensible under all three tolerance scenarios.

Other details are choice-dependent. For $\zeta = 0.01$, the number of runs labelled `stable_response` is 0, 8, or 18 under the strict, baseline, and permissive scenarios, respectively; the strict scenario also reclassifies 8 runs at $\zeta = 0.03$ as `stable_amplitude_with_residual`. Because both tolerances were changed together, this comparison cannot identify which criterion caused each reclassification.

Estimator choice changes little at the sampled frequencies, but not literally nothing. At $\zeta = 0.4$, peak-to-peak selects frequency ratio $r = 0.83$, whereas demeaned RMS and harmonic least-squares select $r = 0.82$. The difference $\Delta r = 0.01$ is exactly one frequency-grid step, so the data do not support a more precise peak location. Finally, the close estimator agreement at $\zeta = 0.01$ is **not** evidence that the response has settled: all three methods can summarize the same transient-contaminated record.

## Conclude the investigation

Use the expected-evidence checklist in [`SCENARIO.md`](SCENARIO.md) to assemble the final argument. Separate supported resonance trends from the unresolved lightly damped runs, state the sampled frequency resolution, and report which conclusions survived the sensitivity analysis.